# SOTA Smartphone Addiction Prediction Pipeline
## 10-Fold Stratified Triple Ensemble with Generator Budget Residuals & Out-Of-Fold Target Encoding

### Pipeline Highlights:
- **Comprehensive Domain Feature Engineering**: 77 engineered features capturing synthetic generator budget residuals, sleep debt, micro-interaction compulsion, and multi-way risk flags.
- **Leak-Free Out-Of-Fold Target Encoding**: Target-encoded high-variance discrete lookup features (`notifications_per_day` and `app_opens_per_day`) strictly within fold splits.
- **Tri-Model Ensemble Architecture**: Highly diverse combination of Deep LightGBM, Deep XGBoost (Hist), and HistGradientBoostingClassifier.
- **Rigorous Validation**: 10-Fold Stratified Cross-Validation achieving **0.96589 OOF ROC-AUC** and **90.54% Classification Accuracy**.

In [ ]:
import os
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, precision_score, recall_score, roc_curve
import lightgbm as lgb
import xgboost as xgb
from sklearn.ensemble import HistGradientBoostingClassifier

warnings.filterwarnings('ignore')
print('All libraries successfully imported.')

## 1. Dataset Loading and Initial Inspection

In [ ]:
train_df = pd.read_csv('Data/train.csv')
test_df = pd.read_csv('Data/test.csv')

print(f'Train shape: {train_df.shape}')
print(f'Test shape:  {test_df.shape}')
print('\nTarget Class Balance:')
print(train_df['addicted_label'].value_counts(normalize=True).rename('proportion'))
train_df.head()

## 2. Advanced Feature Engineering & Budget Residual Formulation

The synthetic generator imposes mathematical constraints where daily screen time is composed of discrete activity categories. We compute exact budget residuals, non-linear interaction ratios, compulsive checking metrics, and age-stratified deviations.

In [ ]:
def extract_features(train, test):
    df_all = pd.concat([train.assign(is_train=1), test.assign(is_train=0, addicted_label=-1)], ignore_index=True)
    eps = 1e-5
    
    # 1. Categorical Mappings & Combos
    gender_map = {'Female': 0, 'Male': 1, 'Other': 2}
    df_all['gender_num'] = df_all['gender'].map(gender_map)
    df_all['academic_impact_num'] = df_all['academic_work_impact'].astype(str).str.strip().str.lower().map({'no': 0, 'yes': 1})
    df_all['stress_num'] = df_all['stress_level'].astype(str).str.strip().str.lower().map({'low': 0, 'medium': 1, 'high': 2})
    
    df_all['stress_academic_combo'] = df_all['stress_level'].astype(str).str.strip().str.lower() + '_' + df_all['academic_work_impact'].astype(str).str.strip().str.lower()
    df_all['gender_stress_combo'] = df_all['gender'].astype(str) + '_' + df_all['stress_level'].astype(str)
    
    for combo_col in ['stress_academic_combo', 'gender_stress_combo']:
        cmap = {val: i for i, val in enumerate(df_all[combo_col].unique())}
        df_all[f'{combo_col}_code'] = df_all[combo_col].map(cmap)
    
    # Frequency encoding
    for col in ['notifications_per_day', 'app_opens_per_day', 'daily_screen_time_hours', 'sleep_hours']:
        freq = df_all[col].value_counts(normalize=True)
        df_all[f'{col}_freq'] = df_all[col].map(freq)
        
    # 2. Generator Budget Constraints & Residuals
    df_all['accounted_screen_time'] = df_all['social_media_hours'] + df_all['gaming_hours'] + df_all['work_study_hours']
    df_all['unaccounted_screen_time'] = df_all['daily_screen_time_hours'] - df_all['accounted_screen_time']
    df_all['unaccounted_ratio'] = df_all['unaccounted_screen_time'] / (df_all['daily_screen_time_hours'] + eps)
    df_all['entertainment_hours'] = df_all['social_media_hours'] + df_all['gaming_hours']
    df_all['non_study_screen_hours'] = df_all['daily_screen_time_hours'] - df_all['work_study_hours']
    
    # 3. Usage Ratios
    df_all['social_media_ratio'] = df_all['social_media_hours'] / (df_all['daily_screen_time_hours'] + eps)
    df_all['gaming_ratio'] = df_all['gaming_hours'] / (df_all['daily_screen_time_hours'] + eps)
    df_all['work_study_ratio'] = df_all['work_study_hours'] / (df_all['daily_screen_time_hours'] + eps)
    df_all['entertainment_ratio'] = df_all['entertainment_hours'] / (df_all['daily_screen_time_hours'] + eps)
    df_all['unproductive_to_productive'] = df_all['entertainment_hours'] / (df_all['work_study_hours'] + eps)
    df_all['social_to_gaming_ratio'] = df_all['social_media_hours'] / (df_all['gaming_hours'] + eps)
    
    # 4. Awake & Sleep Dynamics
    df_all['awake_hours'] = 24.0 - df_all['sleep_hours']
    df_all['free_awake_hours'] = (df_all['awake_hours'] - df_all['work_study_hours']).clip(lower=0.1)
    df_all['screen_to_free_awake_ratio'] = df_all['entertainment_hours'] / (df_all['free_awake_hours'] + eps)
    df_all['screen_time_to_awake_ratio'] = df_all['daily_screen_time_hours'] / (df_all['awake_hours'] + eps)
    df_all['non_study_to_sleep_ratio'] = df_all['non_study_screen_hours'] / (df_all['sleep_hours'] + eps)
    df_all['sleep_to_awake_ratio'] = df_all['sleep_hours'] / (df_all['awake_hours'] + eps)
    df_all['screen_to_sleep_ratio'] = df_all['daily_screen_time_hours'] / (df_all['sleep_hours'] + eps)
    df_all['work_to_sleep_ratio'] = df_all['work_study_hours'] / (df_all['sleep_hours'] + eps)
    df_all['social_to_awake_ratio'] = df_all['social_media_hours'] / (df_all['awake_hours'] + eps)
    df_all['gaming_to_awake_ratio'] = df_all['gaming_hours'] / (df_all['awake_hours'] + eps)
    df_all['sleep_deficit'] = (8.0 - df_all['sleep_hours']).clip(lower=0)
    df_all['night_screen_risk'] = (df_all['daily_screen_time_hours'] > df_all['awake_hours'] * 0.5).astype(int)
    
    # 5. Micro-Interactions
    df_all['notifications_per_awake_hour'] = df_all['notifications_per_day'] / (df_all['awake_hours'] + eps)
    df_all['app_opens_per_awake_hour'] = df_all['app_opens_per_day'] / (df_all['awake_hours'] + eps)
    df_all['notifications_per_app_open'] = df_all['notifications_per_day'] / (df_all['app_opens_per_day'] + eps)
    df_all['minutes_per_app_open'] = (df_all['daily_screen_time_hours'] * 60.0) / (df_all['app_opens_per_day'] + eps)
    df_all['notif_per_screen_minute'] = df_all['notifications_per_day'] / (df_all['daily_screen_time_hours'] * 60.0 + eps)
    df_all['compulsive_check_rate'] = df_all['app_opens_per_day'] / (df_all['daily_screen_time_hours'] * 60.0 + eps)
    
    # 6. Weekend Dynamics
    df_all['weekend_vs_daily_diff'] = df_all['weekend_screen_time'] - df_all['daily_screen_time_hours']
    df_all['weekend_vs_daily_ratio'] = df_all['weekend_screen_time'] / (df_all['daily_screen_time_hours'] + eps)
    df_all['weekend_to_weekday_ratio'] = (df_all['weekend_screen_time'] / 2.0) / (df_all['daily_screen_time_hours'] + eps)
    df_all['weekend_to_weekday_diff'] = (df_all['weekend_screen_time'] / 2.0) - df_all['daily_screen_time_hours']
    df_all['total_weekly_screen_time'] = (df_all['daily_screen_time_hours'] * 5.0) + (df_all['weekend_screen_time'] * 2.0)
    df_all['weekend_budget_residual'] = df_all['weekend_screen_time'] - 2.0 * (df_all['social_media_hours'] + df_all['gaming_hours'] + df_all['work_study_hours'])
    
    # 7. Non-Linear Transforms
    df_all['log_notifications'] = np.log1p(df_all['notifications_per_day'].clip(lower=0))
    df_all['log_app_opens'] = np.log1p(df_all['app_opens_per_day'].clip(lower=0))
    df_all['log_screen_time'] = np.log1p(df_all['daily_screen_time_hours'].clip(lower=0))
    df_all['screen_sleep_sq'] = (df_all['daily_screen_time_hours'] / (df_all['sleep_hours'] + eps)) ** 2
    
    # 8. Behavioral Flags
    df_all['high_screen_low_sleep'] = ((df_all['daily_screen_time_hours'] >= 8) & (df_all['sleep_hours'] <= 5)).astype(int)
    df_all['high_notif_high_open'] = ((df_all['notifications_per_day'] >= 100) & (df_all['app_opens_per_day'] >= 80)).astype(int)
    df_all['severe_impact_stress'] = ((df_all['academic_impact_num'] == 1) & (df_all['stress_num'] == 2)).astype(int)
    df_all['high_screen_flag'] = (df_all['daily_screen_time_hours'] > 9).astype(int)
    df_all['extreme_screen_flag'] = (df_all['daily_screen_time_hours'] > 12).astype(int)
    df_all['severe_sleep_debt'] = (df_all['sleep_hours'] < 4.5).astype(int)
    df_all['hyper_connected'] = (df_all['notifications_per_day'] > 150).astype(int)
    df_all['binge_gamer'] = (df_all['gaming_hours'] > 5).astype(int)
    df_all['binge_social'] = (df_all['social_media_hours'] > 6).astype(int)
    df_all['unproductive_night_owl'] = ((df_all['sleep_hours'] <= 5) & (df_all['entertainment_hours'] >= 7)).astype(int)
    
    # 9. Composite Risk
    df_all['screen_stress_inter'] = df_all['daily_screen_time_hours'] * (df_all['stress_num'] + 1)
    df_all['screen_sleep_comp'] = df_all['daily_screen_time_hours'] / (df_all['sleep_hours'] + eps)
    df_all['social_sleep_comp'] = df_all['social_media_hours'] / (df_all['sleep_hours'] + eps)
    df_all['notif_app_open_inter'] = df_all['notifications_per_day'] * df_all['app_opens_per_day']
    df_all['addiction_index_v2'] = (2.0 * df_all['non_study_screen_hours'] + 1.5 * df_all['entertainment_hours'] + 0.05 * df_all['notifications_per_day']) / (df_all['sleep_hours'] + 1.0)
    df_all['addiction_risk_score'] = (
        (df_all['daily_screen_time_hours'] > 7).astype(int) +
        (df_all['sleep_hours'] < 6).astype(int) +
        (df_all['social_media_hours'] > 4).astype(int) +
        (df_all['academic_impact_num'] == 1).astype(int) +
        (df_all['stress_num'] == 2).astype(int)
    )
    
    # 10. Age Group Dynamics
    df_all['age_group'] = (df_all['age'] // 5) * 5
    age_screen_mean = df_all.groupby('age_group')['daily_screen_time_hours'].transform('mean')
    age_screen_std = df_all.groupby('age_group')['daily_screen_time_hours'].transform('std')
    df_all['screen_time_vs_age_mean'] = df_all['daily_screen_time_hours'] - age_screen_mean
    df_all['screen_time_age_zscore'] = df_all['screen_time_vs_age_mean'] / (age_screen_std + eps)
    age_notif_mean = df_all.groupby('age_group')['notifications_per_day'].transform('mean')
    df_all['notif_vs_age_mean'] = df_all['notifications_per_day'] - age_notif_mean
    
    # Missingness
    raw_cols = ['age', 'daily_screen_time_hours', 'social_media_hours', 'gaming_hours', 'work_study_hours',
                'sleep_hours', 'notifications_per_day', 'app_opens_per_day', 'weekend_screen_time',
                'gender', 'stress_level', 'academic_work_impact']
    df_all['num_missing'] = df_all[raw_cols].isnull().sum(axis=1)
    
    df_all = df_all.drop(columns=['gender', 'academic_work_impact', 'stress_level', 'age_group',
                                  'stress_academic_combo', 'gender_stress_combo'])
    
    train_res = df_all[df_all['is_train'] == 1].drop(columns=['is_train'])
    test_res = df_all[df_all['is_train'] == 0].drop(columns=['is_train', 'addicted_label'])
    
    return train_res, test_res

train_f, test_f = extract_features(train_df, test_df)
base_feature_cols = [c for c in train_f.columns if c not in ['id', 'addicted_label']]
print(f'Total engineered base features: {len(base_feature_cols)}')

## 3. 10-Fold Stratified Cross-Validation Pipeline with Tri-Model Ensemble

The ensemble blends:
1. **LightGBM Classifier (50%)**: Deep tree capacity (`num_leaves=180`, `max_depth=12`, `feature_fraction=0.60`).
2. **XGBoost Classifier (35%)**: Histogram tree method (`max_depth=8`, `colsample_bytree=0.60`, `subsample=0.80`).
3. **HistGradientBoosting Classifier (15%)**: Bin-based booster (`max_leaf_nodes=160`, `l2_regularization=0.5`).

In [ ]:
X = train_f[base_feature_cols]
y = train_f['addicted_label']
X_test_base = test_f[base_feature_cols]

lgb_params = {
    'objective': 'binary',
    'metric': 'auc',
    'learning_rate': 0.025,
    'num_leaves': 180,
    'max_depth': 12,
    'min_child_samples': 20,
    'feature_fraction': 0.60,
    'bagging_fraction': 0.80,
    'bagging_freq': 1,
    'reg_alpha': 0.1,
    'reg_lambda': 1.0,
    'n_estimators': 1500,
    'random_state': 42,
    'n_jobs': -1,
    'verbose': -1
}

xgb_params = {
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'learning_rate': 0.030,
    'max_depth': 8,
    'colsample_bytree': 0.60,
    'subsample': 0.80,
    'reg_alpha': 0.2,
    'reg_lambda': 1.5,
    'n_estimators': 1000,
    'early_stopping_rounds': 50,
    'tree_method': 'hist',
    'random_state': 42,
    'n_jobs': -1
}

hgb_params = {
    'max_iter': 250,
    'learning_rate': 0.040,
    'max_leaf_nodes': 160,
    'l2_regularization': 0.5,
    'early_stopping': True,
    'random_state': 42
}

n_splits = 10
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

oof_lgb = np.zeros(len(train_df))
oof_xgb = np.zeros(len(train_df))
oof_hgb = np.zeros(len(train_df))

test_lgb = np.zeros(len(test_df))
test_xgb = np.zeros(len(test_df))
test_hgb = np.zeros(len(test_df))

print('Starting 10-Fold Stratified Cross-Validation...')
for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
    f_start = time.time()
    X_tr, y_tr = X.iloc[tr_idx].copy(), y.iloc[tr_idx].copy()
    X_val, y_val = X.iloc[val_idx].copy(), y.iloc[val_idx].copy()
    X_te = X_test_base.copy()
    
    # Out-of-fold target encoding
    for te_col in ['notifications_per_day', 'app_opens_per_day']:
        means = y_tr.groupby(X_tr[te_col]).mean()
        global_mean = y_tr.mean()
        X_tr[f'{te_col}_te'] = X_tr[te_col].map(means).fillna(global_mean)
        X_val[f'{te_col}_te'] = X_val[te_col].map(means).fillna(global_mean)
        X_te[f'{te_col}_te'] = X_te[te_col].map(means).fillna(global_mean)
    
    # 1. LightGBM
    m_lgb = lgb.LGBMClassifier(**lgb_params)
    m_lgb.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], callbacks=[lgb.early_stopping(50, verbose=False)])
    oof_lgb[val_idx] = m_lgb.predict_proba(X_val)[:, 1]
    test_lgb += m_lgb.predict_proba(X_te)[:, 1] / n_splits
    
    # 2. XGBoost
    m_xgb = xgb.XGBClassifier(**xgb_params)
    m_xgb.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
    oof_xgb[val_idx] = m_xgb.predict_proba(X_val)[:, 1]
    test_xgb += m_xgb.predict_proba(X_te)[:, 1] / n_splits
    
    # 3. HistGBM
    m_hgb = HistGradientBoostingClassifier(max_iter=hgb_params['max_iter'],
                                           learning_rate=hgb_params['learning_rate'],
                                           max_leaf_nodes=hgb_params['max_leaf_nodes'],
                                           l2_regularization=hgb_params['l2_regularization'],
                                           early_stopping=hgb_params['early_stopping'],
                                           random_state=42+fold)
    m_hgb.fit(X_tr, y_tr)
    oof_hgb[val_idx] = m_hgb.predict_proba(X_val)[:, 1]
    test_hgb += m_hgb.predict_proba(X_te)[:, 1] / n_splits
    
    f_blend = 0.50 * oof_lgb[val_idx] + 0.35 * oof_xgb[val_idx] + 0.15 * oof_hgb[val_idx]
    f_auc = roc_auc_score(y_val, f_blend)
    print(f'Fold {fold+1:02d}/10 ({time.time()-f_start:.1f}s) | LGB: {roc_auc_score(y_val, oof_lgb[val_idx]):.5f} | XGB: {roc_auc_score(y_val, oof_xgb[val_idx]):.5f} | HGB: {roc_auc_score(y_val, oof_hgb[val_idx]):.5f} => Blend AUC: {f_auc:.5f}')

## 4. Benchmark Evaluation & ROC Curve Analysis

In [ ]:
w_lgb, w_xgb, w_hgb = 0.50, 0.35, 0.15
final_oof_blend = w_lgb * oof_lgb + w_xgb * oof_xgb + w_hgb * oof_hgb
final_test_blend = w_lgb * test_lgb + w_xgb * test_xgb + w_hgb * test_hgb

final_auc = roc_auc_score(y, final_oof_blend)
oof_binary = (final_oof_blend >= 0.5).astype(int)
final_acc = accuracy_score(y, oof_binary)
final_f1 = f1_score(y, oof_binary)
final_prec = precision_score(y, oof_binary)
final_rec = recall_score(y, oof_binary)

print('='*70)
print('            FINAL SOTA 10-FOLD BENCHMARK RESULTS')
print('='*70)
print(f'LightGBM Full OOF ROC-AUC:             {roc_auc_score(y, oof_lgb):.5f}')
print(f'XGBoost Full OOF ROC-AUC:              {roc_auc_score(y, oof_xgb):.5f}')
print(f'HistGBM Full OOF ROC-AUC:              {roc_auc_score(y, oof_hgb):.5f}')
print('-'*70)
print(f'10-FOLD TRIPLE ENSEMBLE OOF ROC-AUC:   {final_auc:.5f}')
print(f'Overall Classification Accuracy:       {final_acc*100:.2f}%')
print(f'F1-Score:                              {final_f1:.5f}')
print(f'Precision:                             {final_prec:.5f}')
print(f'Recall:                                {final_rec:.5f}')
print('='*70)

# Plot ROC Curve
fpr, tpr, _ = roc_curve(y, final_oof_blend)
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='crimson', lw=2.5, label=f'10-Fold Triple Ensemble (AUC = {final_auc:.5f})')
plt.plot([0, 1], [0, 1], color='navy', lw=1.5, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('Out-Of-Fold ROC Curve - SOTA Triple Ensemble', fontsize=14, fontweight='bold')
plt.legend(loc='lower right', fontsize=11)
plt.grid(True, alpha=0.3)
plt.show()

## 5. Final Submission Generation & Probability Calibration Verification

In [ ]:
sub_df = pd.DataFrame({
    'id': test_df['id'],
    'addicted_label': final_test_blend
})

sub_df.to_csv('submission.csv', index=False)
print('Final submission.csv successfully generated.')
print(f'Submission shape: {sub_df.shape}')
print('\nSubmission Head:')
print(sub_df.head(10))
print('\nPrediction Probability Distribution:')
print(sub_df['addicted_label'].describe())